<a href="https://colab.research.google.com/github/Kshitij-Panday/Aiml_Lab/blob/main/A_Batch_B4_65_KshitijPanday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from collections import deque   # For BFS
import heapq  # For Greedy and A*


# Graph Representation
# Format:

graph = {
    'S': [('A', 1), ('B', 4), ('C', 3)],
    'A': [('D', 5), ('E', 12)],
    'B': [('E', 4), ('F', 5)],
    'C': [('F', 2), ('H', 6)],
    'D': [('G', 5)],
    'E': [('I', 3)],
    'F': [('I', 2), ('J', 4)],
    'H': [('J', 5)],
    'I': [('G', 3)],
    'J': [('G', 4)],
    'G': []   # Goal node
}


# Heuristic values h(n)

heuristic = {
    'S': 10,
    'A': 9,
    'B': 9,
    'C': 6,
    'D': 5,
    'E': 7,
    'F': 4,
    'H': 7,
    'I': 3,
    'J': 6,
    'G': 0
}

In [ ]:
#Q-1 : BFS ==> BFS uses a queue to explore nodes level by level, ensuring the minimum number of edges to reach the goal.
     # It ignores edge costs during search and guarantees the shortest path in terms of hops.
 # After reaching the goal, the path cost is calculated separately.

In [ ]:
def bfs(graph, start, goal):

    # Create queue for BFS
    queue = deque([start])

    # Parent dictionary for path reconstruction
    parent = {start: None}

    # Visited set to avoid revisiting nodes
    visited = set([start])

    # BFS loop
    while queue:
        node = queue.popleft()   # Remove first element (FIFO)

        # If goal found, stop
        if node == goal:
            break

        # Expand neighbors in alphabetical order
        for neighbor, cost in sorted(graph[node]):

            # If neighbor not visited
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                queue.append(neighbor)

    # Reconstruct path
    path = []
    node = goal
    while node:
        path.append(node)
        node = parent[node]
    path.reverse()

    # Calculate total cost
    total_cost = 0
    for i in range(len(path)-1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i+1]:
                total_cost += cost

    return path, total_cost


# Run BFS
path, cost = bfs(graph, 'S', 'G')
print("BFS Path:", path)
print("Total Cost:", cost)
print("Number of Edges:", len(path)-1)

BFS Path: ['S', 'A', 'D', 'G']
Total Cost: 11
Number of Edges: 3


In [ ]:
#Q-2 : DFS ==>DFS uses a stack (LIFO) to explore one branch deeply before backtracking.
# It does not guarantee minimum edges or minimum cost but finds a path quickly.
# It depends on expansion order.

In [ ]:
def dfs(graph, start, goal):

    # Stack for DFS
    stack = [start]

    parent = {start: None}
    visited = set()

    # DFS loop
    while stack:
        node = stack.pop()   # LIFO (Last In First Out)

        if node in visited:
            continue

        visited.add(node)

        if node == goal:
            break

        # Reverse alphabetical so smallest comes first when popped
        for neighbor, cost in sorted(graph[node], reverse=True):
            if neighbor not in visited:
                parent[neighbor] = node
                stack.append(neighbor)

    # Reconstruct path
    path = []
    node = goal
    while node:
        path.append(node)
        node = parent[node]
    path.reverse()

    # Calculate cost
    total_cost = 0
    for i in range(len(path)-1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i+1]:
                total_cost += cost

    return path, total_cost


# Run DFS
path, cost = dfs(graph, 'S', 'G')
print("DFS Path:", path)
print("Total Cost:", cost)

DFS Path: ['S', 'A', 'D', 'G']
Total Cost: 11


In [ ]:
#Q-3:BFS vs DFS for low batter : This code checks whether the number of edges in the BFS path is less than or equal to 4.
# Since BFS guarantees minimum hops, it is safest under move constraints.

In [ ]:
# Use BFS result
path, cost = bfs(graph, 'S', 'G')

# Check number of edges
if len(path)-1 <= 4:
    print("Battery constraint satisfied.")
else:
    print("Battery constraint NOT satisfied.")

Battery constraint satisfied.


In [ ]:
# Q-4: Greedy Best First : Greedy search selects the node with the smallest heuristic value h(n).
# It ignores the actual cost already travelled and may produce suboptimal solutions.
# It is fast but not guaranteed optimal.

In [ ]:
def greedy_search(graph, start, goal):

    # Min-heap priority queue
    open_list = []

    # Push start node with heuristic value
    heapq.heappush(open_list, (heuristic[start], start))

    parent = {start: None}
    visited = set()

    while open_list:
        h_value, node = heapq.heappop(open_list)

        if node in visited:
            continue

        visited.add(node)

        if node == goal:
            break

        # Expand neighbors
        for neighbor, cost in graph[node]:
            if neighbor not in visited:
                parent[neighbor] = node
                heapq.heappush(open_list, (heuristic[neighbor], neighbor))

    # Reconstruct path
    path = []
    node = goal
    while node:
        path.append(node)
        node = parent[node]
    path.reverse()

    # Calculate cost
    total_cost = 0
    for i in range(len(path)-1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i+1]:
                total_cost += cost

    return path, total_cost


# Run Greedy
path, cost = greedy_search(graph, 'S', 'G')
print("Greedy Path:", path)
print("Total Cost:", cost)

Greedy Path: ['S', 'C', 'F', 'I', 'G']
Total Cost: 10


In [ ]:
# Q-5: A* ==> A* uses f(n) = g(n) + h(n), combining actual path cost and estimated remaining cost.
# It guarantees optimal solution if the heuristic is admissible.
# It balances exploration and cost efficiency.

In [ ]:
def a_star(graph, start, goal):

    # Min-heap with (f, g, node)
    open_list = []
    heapq.heappush(open_list, (heuristic[start], 0, start))

    parent = {start: None}

    # Store g(n) values
    g_cost = {node: float('inf') for node in graph}
    g_cost[start] = 0

    visited = set()

    while open_list:
        f_value, current_g, node = heapq.heappop(open_list)

        if node in visited:
            continue

        visited.add(node)

        if node == goal:
            break

        # Expand neighbors
        for neighbor, cost in graph[node]:

            # New g value
            new_g = g_cost[node] + cost

            # If better path found
            if new_g < g_cost[neighbor]:
                g_cost[neighbor] = new_g
                f = new_g + heuristic[neighbor]
                heapq.heappush(open_list, (f, new_g, neighbor))
                parent[neighbor] = node

    # Reconstruct path
    path = []
    node = goal
    while node:
        path.append(node)
        node = parent[node]
    path.reverse()

    return path, g_cost[goal]


# Run A*
path, cost = a_star(graph, 'S', 'G')
print("A* Path:", path)
print("Optimal Cost:", cost)

A* Path: ['S', 'C', 'F', 'I', 'G']
Optimal Cost: 10


In [ ]:
# Q-7: Modified Greedy ==> After increasing the cost of edge I → G to 20, greedy still chooses the same path
# because it uses only heuristic values.The total cost increases to 27,showing greedy can fail when actual edge costs change.

In [ ]:
# Make copy of original graph
graph_modified = graph.copy()

# Modify I → G edge cost
graph_modified['I'] = [('G', 20)]

# Run Greedy again
path, cost = greedy_search(graph_modified, 'S', 'G')

print("Greedy Path After Modification:", path)
print("Total Cost After Modification:", cost)

Greedy Path After Modification: ['S', 'C', 'F', 'I', 'G']
Total Cost After Modification: 27
